<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Learning Rate Scheduling</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">La tasa de aprendizaje no tiene por qué mantenerse fija durante todo el entrenamiento. Este notebook introduce los schedulers como mecanismos para modificarla según una estrategia definida.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Ajustar el learning rate en el momento adecuado puede mejorar tanto la velocidad como la calidad de la convergencia.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender por qué conviene cambiar la tasa de aprendizaje y qué problema intenta resolver un scheduler.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Un learning rate útil al principio puede dejar de serlo más adelante.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Un scheduler cambia el valor del learning rate a lo largo del entrenamiento siguiendo una regla o un evento.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas del conjunto de entrenamiento.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en un tensor columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten demostrar cómo el learning rate scheduling modifica la tasa de aprendizaje durante el entrenamiento."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y etiquetas para verificar su forma y contenido.


In [ ]:
# Bloque 2 — Definición de la red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedLRS()" que representa un modelo de red neuronal simple.
La clase "RedLRS()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedLRS()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos la llamada a la función "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos la llamada a la función "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo tras aplicar las capas y la activación a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedLRS()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo el learning rate scheduling afecta la actualización de parámetros."""

import torch.nn as nn

class RedLRS(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedLRS()

red # Muestra la estructura de la red neuronal para verificar su definición.


In [ ]:
# Bloque 3 — Función de pérdida y optimizador
"""Este bloque define la función de pérdida y el optimizador utilizados durante el entrenamiento.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida, que recibe los logits de la red y las etiquetas.

Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.
El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador; en este caso, 0.1.
El optimizador Adam actualiza los parámetros de la red utilizando los gradientes calculados."""

criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.Adam(red.parameters(), lr=0.1)

criterio, optimizador # Muestra los objetos de la función de pérdida y el optimizador para verificar su creación.


In [ ]:
# Bloque 4 — Definición del scheduler
"""Este bloque define el scheduler que ajustará dinámicamente el learning rate.
Aplicamos la clase "torch.optim.lr_scheduler.StepLR()" a la variable "scheduler" para crear un objeto que controla la actualización del learning rate.
El argumento "optimizador" especifica el optimizador cuya tasa de aprendizaje será actualizada por el scheduler.
El parámetro "step_size=" especifica el número de llamadas al scheduler entre reducciones de la tasa de aprendizaje; aquí se llama una vez por época.
El parámetro "gamma=" especifica el factor por el cual se reducirá la tasa de aprendizaje cada vez que se cumpla el "step_size".
La variable "scheduler" almacena el objeto que controla la actualización del learning rate.
El scheduler permite ajustar dinámicamente la tasa de aprendizaje durante el entrenamiento, lo que puede mejorar la convergencia y el 
rendimiento del modelo."""

scheduler = torch.optim.lr_scheduler.StepLR(optimizador, step_size=5, gamma=0.5)

scheduler # Muestra la representación del objeto scheduler.


In [ ]:
# Bloque 5 — Loop de entrenamiento con scheduler
"""Este bloque implementa el loop de entrenamiento con learning rate scheduling.
La variable "epocas" define cuántas veces se repite el ciclo completo; en este caso, 15 veces.
La variable "historial_lr" se inicializa como una lista vacía "[]" para almacenar los valores del learning rate en cada época.
La variable "historial_perdida" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida en cada época.

Utilizamos un bucle "for" para iterar sobre el número de épocas definido en la variable "epocas"; es decir, se repetirá 15 veces en este caso.
La expresión "for _ in range(epocas):" inicia el bucle que se repetirá el número de veces definido en "epocas".

En cada iteración se ejecuta el forward pass, se calcula la pérdida, se reinician los gradientes previos, se ejecuta backward para obtener gradientes,
se actualizan los parámetros con el optimizador y se actualiza el learning rate mediante el scheduler.

Además, en cada iteración se almacenan el learning rate mediante la expresión "optimizador.param_groups[0]['lr']" y la pérdida mediante "perdida.item()".
Después de cada actualización, el scheduler ajusta el learning rate según la regla definida.
La variable "historial_lr" almacena la tasa después del ajuste del scheduler; la pérdida se calculó antes de actualizar los parámetros.
Por ello, cuando cambia la tasa, la registrada corresponde a la siguiente actualización, no a la que acaba de realizarse."""

epocas = 15
historial_lr = []
historial_perdida = []

for _ in range(epocas):
    pred = red(entradas)
    perdida = criterio(pred, etiquetas)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    scheduler.step()

    historial_lr.append(optimizador.param_groups[0]["lr"])
    historial_perdida.append(float(perdida.item()))



In [ ]:
# Bloque 6 — Resultados finales
"""Este bloque organiza los resultados del proceso de learning rate scheduling.
La variable "resultado" almacena en forma de diccionario el historial de tasas de aprendizaje y las pérdidas por época.
La estructura permite visualizar cómo el scheduler ajustó el learning rate durante el entrenamiento."""

resultado = {
    "learning_rates": historial_lr,
    "perdidas_por_epoca": historial_perdida
}

resultado # Muestra el diccionario con los resultados finales.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo adaptar la tasa de aprendizaje según el momento del entrenamiento.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Mover el learning rate con criterio puede ayudar tanto a explorar al principio como a afinar al final.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Interpretar el scheduler como una solución mágica sin revisar primero si el learning rate base es razonable.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Weight Decay para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Weight Decay y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">07_Optimizacion_avanzada/03_El_weight_decay.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">